# 🏠 WFH Employee Burnout — Complete ML Pipeline
### Dataset: work_from_home_burnout_dataset.csv
### Tasks: Regression (burnout_score) + Classification (burnout_risk)
---

## 📦 Step 1 — Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Splitting & Tuning
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV, RandomizedSearchCV

# Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, VotingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score,
    recall_score, precision_score, f1_score,
    matthews_corrcoef, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')
print('✅ All libraries imported!')

## 📁 Step 2 — Load Data

In [2]:
df = pd.read_csv('./Dataset/wfh_burnout.csv')
print('Shape:', df.shape)
df.head()

Shape: (1800, 11)


,user_id,day_type,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,burnout_score,burnout_risk
0,1,Weekday,9.59,11.86,4,2,0,7.55,91.2,19.17,Low
1,1,Weekend,7.38,10.33,4,1,0,6.69,82.0,29.70,Low
2,1,Weekend,6.31,8.92,1,2,0,8.87,80.6,32.93,Low
3,1,Weekday,8.34,10.70,4,1,1,8.13,70.0,45.47,Low
4,1,Weekend,6.97,9.83,1,2,0,5.85,67.1,51.61,Low


## 🔎 Step 3 — Data Exploration

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\nNull Values:')
print(df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
print('\nDescriptive Stats:')
df.describe()

In [ ]:
# Outlier Detection — IQR Method
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
outlier_counts = []
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    outlier_counts.append([col, n, round(n/len(df)*100, 2)])

outlier_df = pd.DataFrame(outlier_counts, columns=[
                          'Feature', 'Outliers', 'Percentage'])
print(outlier_df.sort_values('Outliers', ascending=False))

## 📊 Step 4 — Visualization (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Burnout Risk Count
sns.countplot(data=df, x='burnout_risk', hue='burnout_risk',
              palette='Accent', legend=False, ax=axes[0, 0])
axes[0, 0].set_title('Burnout Risk Distribution')

# 2. Pie Chart
rc = df['burnout_risk'].value_counts()
axes[0, 1].pie(rc, labels=rc.index, autopct='%1.1f%%')
axes[0, 1].set_title('Burnout Risk Proportion')

# 3. Correlation Heatmap
num_df = df.select_dtypes(
    include=['float64', 'int64']).drop(columns=['user_id'])
sns.heatmap(num_df.corr(), annot=True,
            cmap='coolwarm', fmt='.2f', ax=axes[0, 2])
axes[0, 2].set_title('Correlation Matrix')

# 4. Boxplot
sns.boxplot(data=df, x='after_hours_work',
            y='burnout_score', hue='day_type', ax=axes[1, 0])
axes[1, 0].set_title('After-Hours Work vs Burnout Score')

# 5. Regression Plot
sns.regplot(data=df, x='work_hours', y='burnout_score',
            scatter_kws={'alpha': 0.4}, line_kws={'color': 'red'}, ax=axes[1, 1])
axes[1, 1].set_title('Work Hours vs Burnout Score')

# 6. Sleep vs Burnout
sns.scatterplot(data=df, x='sleep_hours', y='burnout_score',
                hue='burnout_risk', palette='Set2', ax=axes[1, 2])
axes[1, 2].set_title('Sleep Hours vs Burnout Score')

plt.tight_layout()
plt.show()

In [ ]:
# Violin Plots
fig, axes = plt.subplots(1, 5, figsize=(22, 6))
plots = [
    ('day_type', 'burnout_score', 'Burnout by Day Type'),
    ('burnout_risk', 'burnout_score', 'Burnout Score by Risk'),
    ('burnout_risk', 'work_hours', 'Work Hours by Risk'),
    ('burnout_risk', 'sleep_hours', 'Sleep Hours by Risk'),
    ('after_hours_work', 'burnout_score', 'After-Hours vs Burnout'),
]
for ax, (x, y, title) in zip(axes, plots):
    sns.violinplot(data=df, x=x, y=y, palette='Accent',
                   inner='quartile', ax=ax)
    ax.set_title(title, fontsize=10)
plt.tight_layout()
plt.show()

## ⚙️ Step 5 — Feature Engineering

In [ ]:
df_feat = df.copy()
df_feat['productivity_per_hour'] = np.where(
    df_feat['work_hours'] != 0,
    np.round(df_feat['task_completion_rate'] /
             df_feat['work_hours'], 2), np.nan
)
df_feat['workload'] = df_feat['work_hours'] + \
    df_feat['meetings_count'] + df_feat['screen_time_hours']
df_feat['sleep_deficit'] = np.round(
    np.maximum(0, 8 - df_feat['sleep_hours']), 2)
print('New features: productivity_per_hour, workload, sleep_deficit')
df_feat[['productivity_per_hour', 'workload', 'sleep_deficit']].describe()

## 🧹 Step 6 — Preprocessing

In [ ]:
df_clean = df_feat.drop(columns=['user_id'])

le = LabelEncoder()
df_clean['day_type'] = le.fit_transform(df_clean['day_type'])

risk_map = {'Low': 0, 'Medium': 1, 'High': 2}
df_clean['burnout_risk'] = df_clean['burnout_risk'].map(risk_map)

print('Encoding done!')
print(df_clean.head())

---
# 🔢 PART A — REGRESSION (Predict burnout_score)
---

In [ ]:
X_reg = df_clean.drop(columns=['burnout_score', 'burnout_risk'])
y_reg = df_clean['burnout_score']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

scaler_r = RobustScaler()
X_tr_sc = scaler_r.fit_transform(X_tr)
X_te_sc = scaler_r.transform(X_te)

reg_records = []

# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_tr_sc, y_tr)
lr_r2 = r2_score(y_te, lr.predict(X_te_sc))
reg_records.append({'Model': 'Linear Regression', 'R2': round(
    lr_r2, 4), 'MAE': round(mean_absolute_error(y_te, lr.predict(X_te_sc)), 4)})
print(f'Linear Regression R2: {lr_r2:.4f}')

# 2. Gradient Boosting Regressor
gb_r = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
gb_r.fit(X_tr_sc, y_tr)
gb_r2 = r2_score(y_te, gb_r.predict(X_te_sc))
reg_records.append({'Model': 'GB Regressor', 'R2': round(
    gb_r2, 4), 'MAE': round(mean_absolute_error(y_te, gb_r.predict(X_te_sc)), 4)})
print(f'GB Regressor R2     : {gb_r2:.4f}')

# 3. XGBoost Regressor
xgb_r = XGBRegressor(n_estimators=500, learning_rate=0.01,
                     max_depth=3, reg_lambda=1, random_state=42)
xgb_r.fit(X_tr_sc, y_tr)
xgb_r2 = r2_score(y_te, xgb_r.predict(X_te_sc))
reg_records.append({'Model': 'XGBoost Regressor', 'R2': round(
    xgb_r2, 4), 'MAE': round(mean_absolute_error(y_te, xgb_r.predict(X_te_sc)), 4)})
print(f'XGBoost Regressor R2: {xgb_r2:.4f}')

In [ ]:
# Regression Results
reg_df = pd.DataFrame(reg_records).sort_values('R2', ascending=False)
print(reg_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=reg_df, x='Model', y='R2', palette='Blues_d')
plt.title('Regression Models — R2 Score')
plt.ylim(0.8, 1.0)
plt.tight_layout()
plt.show()

In [ ]:
# Regression → Classification Conversion (XGBoost best model)
def map_score_to_risk(score):
    if score < 45:
        return 0  # Low
    elif score < 95:
        return 1  # Medium
    else:
        return 2  # High


y_pred_final = [map_score_to_risk(s) for s in xgb_r.predict(X_te_sc)]
y_test_final = [map_score_to_risk(s) for s in y_te]

print(' Classification Report (After XGBoost Regression) ')
print(classification_report(y_test_final, y_pred_final,
      target_names=['Low', 'Medium', 'High']))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test_final, y_pred_final), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'Medium', 'High'], yticklabels=['Low', 'Medium', 'High'])
plt.title('Confusion Matrix — Regression → Classification')
plt.show()

---
#  PART B — CLASSIFICATION (Predict burnout_risk)
---

In [ ]:
X_clf = df_clean.drop(columns=['burnout_score', 'burnout_risk'])
y_clf = df_clean['burnout_risk']

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

scaler_c = MinMaxScaler()
X_tr_c_sc = scaler_c.fit_transform(X_tr_c)
X_te_c_sc = scaler_c.transform(X_te_c)

clf_records = []

In [ ]:
# B1 — Boosting Models
print(' B1: Boosting Models ')

gbc = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3)
gbc.fit(X_tr_c_sc, y_tr_c)
gbc_acc = accuracy_score(y_te_c, gbc.predict(X_te_c_sc))
clf_records.append({'Model': 'Gradient Boosting',
                   'Accuracy': round(gbc_acc, 4)})
print(f'GBC     : {gbc_acc:.4f}')

xgb_c = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                      subsample=0.8, colsample_bytree=0.8, eval_metric='logloss')
xgb_c.fit(X_tr_c_sc, y_tr_c)
xgb_acc = accuracy_score(y_te_c, xgb_c.predict(X_te_c_sc))
clf_records.append({'Model': 'XGBoost', 'Accuracy': round(xgb_acc, 4)})
print(f'XGBoost : {xgb_acc:.4f}')

cat = CatBoostClassifier(iterations=300, learning_rate=0.05,
                         depth=6, l2_leaf_reg=3.0, verbose=False)
cat.fit(X_tr_c_sc, y_tr_c)
cat_acc = accuracy_score(y_te_c, cat.predict(X_te_c_sc))
clf_records.append({'Model': 'CatBoost', 'Accuracy': round(cat_acc, 4)})
print(f'CatBoost: {cat_acc:.4f}')

In [ ]:
# B2 — SVM with 5-Fold Cross Validation
print(' B2: SVM + 5-Fold CV ')


def perf_eval(y_true, y_prob):
    yp = np.argmax(y_prob, axis=1)
    ACC = accuracy_score(y_true, yp)
    REC = recall_score(y_true, yp, average='macro', zero_division=0)
    PRE = precision_score(y_true, yp, average='macro', zero_division=0)
    MCC = matthews_corrcoef(y_true, yp)
    F1 = f1_score(y_true, yp, average='macro', zero_division=0)
    AUC = roc_auc_score(y_true, y_prob, multi_class='ovr')
    return [round(x, 4) for x in [ACC, REC, PRE, MCC, F1, AUC]]


svm_model = SVC(probability=True)
svm_model.fit(X_tr_c_sc, y_tr_c)

yp_tr_prob = cross_val_predict(
    svm_model, X_tr_c_sc, y_tr_c, cv=5, method='predict_proba')
yp_te_prob = svm_model.predict_proba(X_te_c_sc)

cols = ['ACC', 'Recall', 'Precision', 'MCC', 'F1', 'AUC']
tr_m = perf_eval(y_tr_c, yp_tr_prob)
te_m = perf_eval(y_te_c, yp_te_prob)

print(f'Metrics : {cols}')
print(f'Train   : {tr_m}')
print(f'Test    : {te_m}')
clf_records.append({'Model': 'SVM (5-Fold CV)', 'Accuracy': te_m[0]})

In [ ]:
# B3 — RandomizedSearchCV Hyperparameter Tuning
print(' B3: RandomizedSearchCV ')

param_gb = {'n_estimators': np.arange(
    100, 500, 50), 'learning_rate': np.logspace(-3, -1, 20), 'max_depth': np.arange(3, 8)}
param_xgb = {'n_estimators': np.arange(100, 500, 50), 'learning_rate': np.logspace(
    -3, -1, 20), 'max_depth': np.arange(3, 10), 'subsample': np.linspace(0.6, 1.0, 5)}

rand_gbc = RandomizedSearchCV(GradientBoostingClassifier(
), param_gb, n_iter=20, cv=5, scoring='accuracy', n_jobs=-1, random_state=42, verbose=0)
rand_xgb = RandomizedSearchCV(XGBClassifier(eval_metric='logloss'), param_xgb,
                              n_iter=20, cv=5, scoring='accuracy', n_jobs=-1, random_state=42, verbose=0)

rand_gbc.fit(X_tr_c_sc, y_tr_c)
rand_xgb.fit(X_tr_c_sc, y_tr_c)

tuned_gbc_acc = accuracy_score(y_te_c, rand_gbc.predict(X_te_c_sc))
tuned_xgb_acc = accuracy_score(y_te_c, rand_xgb.predict(X_te_c_sc))

print(f'Best GBC params : {rand_gbc.best_params_}')
print(f'Best XGB params : {rand_xgb.best_params_}')
print(f'Tuned GBC Acc   : {tuned_gbc_acc:.4f}')
print(f'Tuned XGB Acc   : {tuned_xgb_acc:.4f}')

clf_records.append({'Model': 'Tuned GBC (RandomSearch)',
                   'Accuracy': round(tuned_gbc_acc, 4)})
clf_records.append({'Model': 'Tuned XGB (RandomSearch)',
                   'Accuracy': round(tuned_xgb_acc, 4)})

In [ ]:
# B4 — Ensemble Voting Classifier (GridSearchCV)
print(' B4: Ensemble Voting + GridSearchCV ')

eclf = VotingClassifier(
    estimators=[('lr', LogisticRegression(random_state=42)),
                ('et', ExtraTreesClassifier(random_state=42)),
                ('svm', SVC(probability=True, random_state=42))],
    voting='soft'
)
params = {'lr__C': [1.0, 10.0],
          'et__n_estimators': [50, 100], 'svm__C': [4, 8]}

best_eclf = GridSearchCV(eclf, param_grid=params, cv=5,
                         scoring='accuracy', n_jobs=-1, verbose=0)
best_eclf.fit(X_tr_c_sc, y_tr_c)

final_model = best_eclf.best_estimator_
final_model.fit(X_tr_c_sc, y_tr_c)

yp_tr_vc = cross_val_predict(
    final_model, X_tr_c_sc, y_tr_c, cv=5, method='predict_proba')
yp_te_vc = final_model.predict_proba(X_te_c_sc)

vc_tr = perf_eval(y_tr_c, yp_tr_vc)
vc_te = perf_eval(y_te_c, yp_te_vc)

print(f'Best Params: {best_eclf.best_params_}')
print(f'Train: ACC={vc_tr[0]}, F1={vc_tr[4]}, AUC={vc_tr[5]}')
print(f'Test : ACC={vc_te[0]}, F1={vc_te[4]}, AUC={vc_te[5]}')
clf_records.append(
    {'Model': 'Ensemble Voting (GridSearch)', 'Accuracy': vc_te[0]})

In [ ]:
# B5 — Pipeline: 5 Classic Models
print(' B5: Pipeline Models ')

X_raw = df_clean.drop(columns=['burnout_score', 'burnout_risk'])
y_raw = df_clean['burnout_risk']
num_c = X_raw.select_dtypes(include=['float64', 'int64']).columns
cat_c = X_raw.select_dtypes(include=['object', 'category']).columns

pre = ColumnTransformer([
    ('num', StandardScaler(), num_c),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_c)
])

X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw)

pipe_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest':       RandomForestClassifier(random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'SVM (Pipeline)':      SVC(),
    'KNN':                 KNeighborsClassifier()
}

for name, model in pipe_models.items():
    pipe = Pipeline([('pre', pre), ('model', model)])
    pipe.fit(X_tr_p, y_tr_p)
    acc = accuracy_score(y_te_p, pipe.predict(X_te_p))
    clf_records.append({'Model': name, 'Accuracy': round(acc, 4)})
    print(f'{name:25s}: {acc:.4f}')

## 🏆 Final Results — All Classification Models

In [ ]:
clf_df = pd.DataFrame(clf_records).sort_values(
    'Accuracy', ascending=False).reset_index(drop=True)
clf_df.index += 1
print(clf_df.to_string())

plt.figure(figsize=(14, 6))
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(clf_df))]
bars = plt.barh(clf_df['Model'], clf_df['Accuracy'], color=colors)
plt.xlabel('Accuracy')
plt.title(' WFH Burnout — All Classification Models Compared')
plt.xlim(0.85, 1.0)
for bar, val in zip(bars, clf_df['Accuracy']):
    plt.text(bar.get_width()+0.001, bar.get_y()+bar.get_height() /
             2, f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

best = clf_df.iloc[0]
print(f'\n Best Model: {best["Model"]} — Accuracy: {best["Accuracy"]}')

---
# ✅ Summary — WFH Burnout Dataset

| Part | Task | Target | Models |
|------|------|--------|--------|
| A | Regression | burnout_score | Linear, GB, XGBoost |
| B1 | Classification | burnout_risk | GBC, XGBoost, CatBoost |
| B2 | SVM + 5-Fold CV | burnout_risk | SVC |
| B3 | Hyperparameter Tuning | burnout_risk | RandomizedSearchCV |
| B4 | Ensemble Voting | burnout_risk | LR + ET + SVM (GridSearch) |
| B5 | Pipeline Models | burnout_risk | LR, RF, DT, SVM, KNN |
---